In [3]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Load dataset
movie_file = pd.read_csv('prediction.csv', encoding='latin1')

# Clean missing values
movie_file.dropna(inplace=True)

# Fix regex warnings
movie_file['Duration'] = movie_file['Duration'].str.extract(r'(\d+)')
movie_file['Duration'] = pd.to_numeric(movie_file['Duration'], errors='coerce')

movie_file['Year'] = movie_file['Year'].str.extract(r'(\d+)').astype(int)

movie_file['Votes'] = movie_file['Votes'].str.replace(',', '').astype(int)

# Feature engineering
movie_file['Actor'] = movie_file['Actor 1'] + ', ' + movie_file['Actor 2'] + ', ' + movie_file['Actor 3']

movie_file['Directors'] = movie_file['Director'].astype('category').cat.codes
movie_file['Genres'] = movie_file['Genre'].astype('category').cat.codes
movie_file['Actors'] = movie_file['Actor'].astype('category').cat.codes

# Input / Output
X = movie_file.drop(['Name', 'Genre', 'Rating', 'Director', 'Actor 1', 'Actor 2', 'Actor 3', 'Actor'], axis=1)
Y = movie_file['Rating']

# Train test split
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=1)

# Evaluation function
def evaluate(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

def evaluate(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)

    return r2, rmse, mae
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

# Models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "XGBoost": XGBRegressor(),
    "LightGBM": LGBMRegressor(),
    "KNN": KNeighborsRegressor()
}

results = []

# Train models
for name, model in models.items():
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    r2, rmse, mae = evaluate(y_test, preds)
    results.append([name, r2, rmse, mae])

# Results table
results_df = pd.DataFrame(results, columns=["Model", "R2 Score", "RMSE", "MAE"])
results_df = results_df.sort_values(by="R2 Score", ascending=False)

print(results_df)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000429 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1207
[LightGBM] [Info] Number of data points in the train set: 4527, number of used features: 6
[LightGBM] [Info] Start training from score 5.910029
               Model  R2 Score      RMSE       MAE
5           LightGBM  0.380307  1.070609  0.805919
3  Gradient Boosting  0.368929  1.080393  0.826440
1      Random Forest  0.344001  1.101524  0.833647
4            XGBoost  0.317183  1.123815  0.835640
0  Linear Regression  0.098704  1.291148  1.027925
6                KNN -0.009562  1.366498  1.081466
2      Decision Tree -0.274406  1.535312  1.156272
